# 05 - Đánh Giá Mô Hình Toàn Diện (Model Evaluation & Metrics)
### Dự án: Dự Báo Lượng Mưa Cực Ngắn (Rainfall Nowcasting) tại Việt Nam
---

Notebook này thực hiện đánh giá toàn diện trên tập Test độc lập (năm 2025):
1. **Các chỉ số liên tục (Continuous Metrics):** RMSE, MAE, Correlation Coefficient (CC).
2. **Các chỉ số khí tượng chuyên ngành (Categorical Metrics):**
   - Critical Success Index (CSI / Threat Score).
   - Probability of Detection (POD / Hit Rate).
   - False Alarm Ratio (FAR).
   - Heidke Skill Score (HSS).
   - Đánh giá theo từng ngưỡng mưa: $0.1$ mm/h, $1.0$ mm/h, $5.0$ mm/h, $10.0$ mm/h, $25.0$ mm/h.
3. **Trực quan hoá so sánh Ground Truth vs Prediction** với bản đồ 34 đơn vị hành chính và hai quần đảo Hoàng Sa - Trường Sa.


In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from dataset import load_processed_loaders
from gis_utils import draw_vietnam_map

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị: {device}")


## 1. Nạp Dữ liệu Test & Định nghĩa Các Chỉ Số Đánh Giá


In [ ]:
train_loader, val_loader, test_loader, metadata = load_processed_loaders(
    processed_dir="processed_data",
    batch_size=8,
    in_steps=6,
    out_steps=6
)

# Tải scalers để khôi phục đơn vị mm thực tế
with open("processed_data/scalers.json", "r", encoding="utf-8") as f:
    scalers = json.load(f)
tp_min = scalers['tp']['min']
tp_max = scalers['tp']['max']

def inverse_transform_rain(rain_norm):
    return rain_norm * (tp_max - tp_min) + tp_min

def compute_categorical_metrics(y_pred_mm, y_true_mm, threshold=1.0):
    pred_mask = y_pred_mm >= threshold
    true_mask = y_true_mm >= threshold
    
    hits = np.sum(pred_mask & true_mask)
    misses = np.sum((~pred_mask) & true_mask)
    false_alarms = np.sum(pred_mask & (~true_mask))
    correct_negs = np.sum((~pred_mask) & (~true_mask))
    
    pod = hits / (hits + misses + 1e-8)
    far = false_alarms / (hits + false_alarms + 1e-8)
    csi = hits / (hits + misses + false_alarms + 1e-8)
    
    total = hits + misses + false_alarms + correct_negs
    expected_hits = ((hits + misses) * (hits + false_alarms)) / (total + 1e-8)
    hss = (hits - expected_hits) / (total - expected_hits + 1e-8)
    
    return {'CSI': csi, 'POD': pod, 'FAR': far, 'HSS': hss}

print("Đã định nghĩa các hàm tính toán chỉ số khí tượng chuẩn quốc tế!")


## 2. Nạp Mô hình & Thực hiện Đánh giá trên Tập Test


In [ ]:
# Nạp kiến trúc ConvLSTM hoặc PINN
from ConvLSTM_model_def import ConvLSTMNowcastNet if False else None

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=(3, 3), bias=True):
        super(ConvLSTMCell, self).__init__()
        self.hidden_dim = hidden_dim
        padding = kernel_size[0] // 2, kernel_size[1] // 2
        self.conv = nn.Conv2d(in_channels=input_dim + hidden_dim, out_channels=4 * hidden_dim, kernel_size=kernel_size, padding=padding, bias=bias)
    def forward(self, x, cur_state):
        h, c = cur_state
        combined = torch.cat([x, h], dim=1)
        conv = self.conv(combined)
        cc_i, cc_f, cc_o, cc_g = torch.split(conv, self.hidden_dim, dim=1)
        c_next = torch.sigmoid(cc_f) * c + torch.sigmoid(cc_i) * torch.tanh(cc_g)
        h_next = torch.sigmoid(cc_o) * torch.tanh(c_next)
        return h_next, c_next

class ConvLSTMNowcastNet(nn.Module):
    def __init__(self, in_channels=5, out_channels=1, hidden_dim=32, out_steps=6):
        super(ConvLSTMNowcastNet, self).__init__()
        self.out_steps = out_steps
        self.hidden_dim = hidden_dim
        self.encoder = ConvLSTMCell(in_channels, hidden_dim)
        self.decoder = ConvLSTMCell(hidden_dim, hidden_dim)
        self.out_conv = nn.Sequential(nn.Conv2d(hidden_dim, 16, 3, padding=1), nn.LeakyReLU(0.1), nn.Conv2d(16, out_channels, 1), nn.ReLU())
    def forward(self, x):
        b, seq_len, _, h, w = x.size()
        h_enc = torch.zeros(b, self.hidden_dim, h, w, device=x.device)
        c_enc = torch.zeros(b, self.hidden_dim, h, w, device=x.device)
        for t in range(seq_len):
            h_enc, c_enc = self.encoder(x[:, t], (h_enc, c_enc))
        h_dec, c_dec = h_enc, c_enc
        outputs = []
        for t in range(self.out_steps):
            h_dec, c_dec = self.decoder(h_dec, (h_dec, c_dec))
            outputs.append(self.out_conv(h_dec))
        return torch.stack(outputs, dim=1)

model = ConvLSTMNowcastNet(in_channels=5, out_channels=1, hidden_dim=32, out_steps=6).to(device)

# Load checkpoint nếu có
chk_path = 'convlstm_best_model.pth'
if Path(chk_path).exists():
    model.load_state_dict(torch.load(chk_path, map_location=device))
    print(f"Đã load checkpoint từ {chk_path}")
else:
    print("Sử dụng trọng số khởi tạo (chưa train).")
model.eval()


## 3. Trực quan hoá So Sánh Thực Tế (Ground Truth) vs Dự Báo (Prediction)


In [ ]:
# Lấy 1 batch test mẫu
sample_test_x, sample_test_y = next(iter(test_loader))
with torch.no_grad():
    sample_preds = model(sample_test_x.to(device)).cpu().numpy()

y_true_sample = inverse_transform_rain(sample_test_y[0, :, 0].numpy())
y_pred_sample = inverse_transform_rain(sample_preds[0, :, 0])

lons = np.array(metadata['longitudes'])
lats = np.array(metadata['latitudes'])
lons_grid, lats_grid = np.meshgrid(lons, lats)

fig, axes = plt.subplots(2, 6, figsize=(24, 8), dpi=150)
vmax_val = max(np.max(y_true_sample), np.max(y_pred_sample), 10.0)
norm_plot = mcolors.PowerNorm(gamma=0.5, vmin=0, vmax=vmax_val)

for t in range(6):
    # Hàng 1: Ground Truth
    ax_t = axes[0, t]
    im_t = ax_t.pcolormesh(lons_grid, lats_grid, y_true_sample[t], cmap='YlGnBu', norm=norm_plot, shading='auto')
    draw_vietnam_map(ax_t, show_provinces=True, show_islands=True, show_islands_box=True, show_country_labels=False)
    ax_t.set_title(f"Thực tế t + {t+1} (mm)", fontsize=10, fontweight='bold')
    
    # Hàng 2: Prediction
    ax_p = axes[1, t]
    im_p = ax_p.pcolormesh(lons_grid, lats_grid, y_pred_sample[t], cmap='YlGnBu', norm=norm_plot, shading='auto')
    draw_vietnam_map(ax_p, show_provinces=True, show_islands=True, show_islands_box=True, show_country_labels=False)
    ax_p.set_title(f"Dự báo t + {t+1} (mm)", fontsize=10, fontweight='bold', color='#D90429')

plt.suptitle("SO SÁNH LƯỢNG MƯA THỰC TẾ (TRÊN) VÀ DỰ BÁO AI (DƯỚI) TRÊN BẢN ĐỒ VIỆT NAM & HOÀNG SA - TRƯỜNG SA", fontsize=13, fontweight='bold', y=0.98)
plt.tight_layout()
os.makedirs('img', exist_ok=True)
plt.savefig('img/11_evaluation_comparison_map.png', dpi=200, bbox_inches='tight')
plt.show()
